In [32]:
import sys
sys.path.append('..')

import pandas as pd
from utils.db_utils import write_table, read_table

In [33]:
# Corrected table name: use 'gdp_per_capita' (capita, not capital)
df = read_table("""
    SELECT *
    FROM sc_silver.gdp_per_capita
    """)

In [34]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (GradientBoostingRegressor, RandomForestRegressor,
                               ExtraTreesRegressor, BaggingRegressor)
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

FORECAST_YEARS = [2024, 2025, 2026, 2027, 2028]
df['year'] = df['date'].astype(str).str[:4].astype(int)

# ─────────────────────────────────────────────
# 2.  MODELS
# ─────────────────────────────────────────────
models = {
    "Gradient Boosting":  GradientBoostingRegressor(n_estimators=300, learning_rate=0.05,
                                                     max_depth=3, random_state=42),
    "Random Forest":      RandomForestRegressor(n_estimators=300, random_state=42),
    "Extra Trees":        ExtraTreesRegressor(n_estimators=300, random_state=42),
    "Poly Ridge (deg 3)": Pipeline([("poly", PolynomialFeatures(degree=3, include_bias=False)),
                                     ("ridge", Ridge(alpha=1.0))]),
    "ElasticNet Poly":    Pipeline([("poly", PolynomialFeatures(degree=2, include_bias=False)),
                                     ("en", ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000))]),
}

# ─────────────────────────────────────────────
# 3.  FIT, EVALUATE, PREDICT PER STATE  ← MODIFIED
#     Now averages predictions across all models
# ─────────────────────────────────────────────
results_rows = []
model_metrics = []   # (state, target, model_name, mae, rmse, r2)

states = df["state"].unique()

for state in states:
    sdf = df[df["state"] == state].sort_values("year").reset_index(drop=True)
    X = sdf[["year"]].values
    X_fut = np.array(FORECAST_YEARS).reshape(-1, 1)

    for target in ["gdp_rm_million", "population"]:
        y = sdf[target].values
        all_preds = []   # ← collect each model's forecast

        for mname, model in models.items():
            # TimeSeriesSplit CV
            tscv = TimeSeriesSplit(n_splits=3)
            scores = cross_val_score(model, X, y, cv=tscv, scoring="r2")
            model.fit(X, y)
            y_pred_train = model.predict(X)
            mae   = mean_absolute_error(y, y_pred_train)
            rmse  = np.sqrt(mean_squared_error(y, y_pred_train))
            r2    = r2_score(y, y_pred_train)
            cv_r2 = scores.mean()
            cv_r2_safe = cv_r2 if not np.isnan(cv_r2) else -999
            model_metrics.append({
                "state": state, "target": target, "model": mname,
                "train_r2": round(r2, 4), "cv_r2": round(cv_r2_safe, 4),
                "mae": round(mae, 2), "rmse": round(rmse, 2)
            })
            all_preds.append(model.predict(X_fut))   # ← store forecast

        # ── Average across all models ──────────────────────────────────
        avg_pred = np.mean(all_preds, axis=0)        # shape: (len(FORECAST_YEARS),)

        for i, yr in enumerate(FORECAST_YEARS):
            results_rows.append({
                "state": state,
                "year": yr,
                "target": target,
                "best_model": "Ensemble Average",    # label changed to reflect averaging
                "predicted_value": round(avg_pred[i], 3),
            })

# Pivot forecasts
gdp_preds = {r["state"]: {} for r in results_rows}
pop_preds  = {r["state"]: {} for r in results_rows}
for r in results_rows:
    if r["target"] == "gdp_rm_million":
        gdp_preds[r["state"]][r["year"]] = r["predicted_value"]
    else:
        pop_preds[r["state"]][r["year"]] = r["predicted_value"]

# Build combined output
output_rows = []
# Historical
for _, row in df.iterrows():
    output_rows.append({
        "state": row["state"],
        "year": int(row["year"]),
        "gdp_rm_million": round(row["gdp_rm_million"], 3),
        "population": int(row["population"]),
        "gdp_per_capita": round(row["gdp_per_capita"], 4),
        "data_type": "historical"
    })
# Forecast
for state in states:
    for yr in FORECAST_YEARS:
        gdp = gdp_preds[state].get(yr, np.nan)
        pop = pop_preds[state].get(yr, np.nan)
        gpc = (gdp * 1e6 / pop) if (gdp and pop) else np.nan
        output_rows.append({
            "state": state,
            "year": yr,
            "gdp_rm_million": round(gdp, 3) if gdp else np.nan,
            "population": int(pop) if pop else np.nan,
            "gdp_per_capita": round(gpc, 4) if gpc else np.nan,
            "data_type": "forecast"
        })

out_df = pd.DataFrame(output_rows).sort_values(["state", "year"]).reset_index(drop=True)
metrics_df = pd.DataFrame(model_metrics)

# ─────────────────────────────────────────────
# 4.  SAVE CSV
# ─────────────────────────────────────────────
out_df.to_csv("malaysia_gdp_forecast.csv", index=False)
metrics_df.to_csv("model_evaluation_metrics.csv", index=False)

# ─────────────────────────────────────────────
# 5.  VISUALISATION  (4 big panels)
# ─────────────────────────────────────────────
HIST_COLOR  = "#4FC3F7"
FORE_COLOR  = "#FF7043"
DARK_BG     = "#0F1117"
CARD_BG     = "#1A1D2E"
GRID_COLOR  = "#2A2D3E"
TEXT_COLOR  = "#E8EAF6"

# ------ Fig 1: Top-5 states – all three KPIs ------
top5 = ["Selangor", "Kuala Lumpur", "Johor", "Sarawak", "Penang"]
colors5 = ["#4FC3F7", "#FF7043", "#66BB6A", "#FFB74D", "#CE93D8"]

fig1, axes = plt.subplots(3, 5, figsize=(22, 13))
fig1.patch.set_facecolor(DARK_BG)
fig1.suptitle("Malaysia State GDP Forecast  ·  Top 5 States  [Ensemble Average]",
              fontsize=18, color=TEXT_COLOR, fontweight="bold", y=1.01)

targets_info = [
    ("gdp_rm_million",  "GDP (RM Million)",   "RM M"),
    ("population",      "Population",          "People"),
    ("gdp_per_capita",  "GDP per Capita (RM)", "RM"),
]

for row_i, (col, label, unit) in enumerate(targets_info):
    for col_i, state in enumerate(top5):
        ax = axes[row_i][col_i]
        ax.set_facecolor(CARD_BG)
        ax.spines[:].set_color(GRID_COLOR)
        ax.tick_params(colors=TEXT_COLOR, labelsize=7)

        hist = out_df[(out_df["state"] == state) & (out_df["data_type"] == "historical")]
        fore = out_df[(out_df["state"] == state) & (out_df["data_type"] == "forecast")]

        ax.plot(hist["year"], hist[col], "o-", color=colors5[col_i],
                linewidth=2, markersize=5, label="Historical")
        ax.plot(fore["year"], fore[col], "s--", color=FORE_COLOR,
                linewidth=2, markersize=5, label="Avg Forecast")  # ← label updated
        ax.axvline(x=2023.5, color="#555577", linewidth=1, linestyle=":")
        ax.fill_between(fore["year"], fore[col]*0.97, fore[col]*1.03,
                        color=FORE_COLOR, alpha=0.15)
        ax.set_title(f"{state}\n{label}", color=TEXT_COLOR, fontsize=8, fontweight="bold")
        ax.grid(True, color=GRID_COLOR, linewidth=0.5, alpha=0.7)
        ax.xaxis.set_tick_params(rotation=45)
        if col_i == 0:
            ax.set_ylabel(unit, color=TEXT_COLOR, fontsize=7)
        if row_i == 0 and col_i == 4:
            ax.legend(fontsize=6, facecolor=DARK_BG, labelcolor=TEXT_COLOR)

plt.tight_layout(pad=1.5)
fig1.savefig("fig1_top5_states.png", dpi=150,
             bbox_inches="tight", facecolor=DARK_BG)
plt.close()

# ------ Fig 2: Model comparison heatmap ------
fig2, axes2 = plt.subplots(1, 2, figsize=(18, 8))
fig2.patch.set_facecolor(DARK_BG)
fig2.suptitle("Model CV R² Score Comparison — All States", fontsize=16,
              color=TEXT_COLOR, fontweight="bold")

for ax_i, tgt in enumerate(["gdp_rm_million", "population"]):
    sub = metrics_df[metrics_df["target"] == tgt]
    pivot = sub.pivot_table(index="state", columns="model", values="cv_r2")
    ax = axes2[ax_i]
    ax.set_facecolor(CARD_BG)
    im = ax.imshow(pivot.values, cmap="RdYlGn", aspect="auto", vmin=-0.2, vmax=1.0)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=35, ha="right", color=TEXT_COLOR, fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, color=TEXT_COLOR, fontsize=8)
    ax.set_title(f"Target: {tgt.replace('_', ' ').title()}", color=TEXT_COLOR, fontsize=11)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    color="white" if val < 0.5 else "black", fontsize=7, fontweight="bold")
    cbar = plt.colorbar(im, ax=ax)
    cbar.ax.tick_params(colors=TEXT_COLOR)

plt.tight_layout(pad=2)
fig2.savefig("fig2_model_heatmap.png", dpi=150,
             bbox_inches="tight", facecolor=DARK_BG)
plt.close()

# ------ Fig 3: GDP per capita 2028 bar chart – all states ------
fig3, ax3 = plt.subplots(figsize=(16, 8))
fig3.patch.set_facecolor(DARK_BG)
ax3.set_facecolor(CARD_BG)

state_gpc_2028 = (out_df[out_df["year"] == 2028]
                  .set_index("state")["gdp_per_capita"]
                  .sort_values(ascending=True))

bar_colors = plt.cm.plasma(np.linspace(0.2, 0.9, len(state_gpc_2028)))
bars = ax3.barh(state_gpc_2028.index, state_gpc_2028.values,
                color=bar_colors, edgecolor="#333", linewidth=0.5)
for bar, val in zip(bars, state_gpc_2028.values):
    ax3.text(val + max(state_gpc_2028)*0.005, bar.get_y() + bar.get_height()/2,
             f"RM {val:,.0f}", va="center", ha="left", color=TEXT_COLOR, fontsize=8)

ax3.set_xlabel("GDP per Capita (RM)", color=TEXT_COLOR, fontsize=10)
ax3.set_title("Projected GDP per Capita in 2028 — All Malaysian States  [Ensemble Average]",
              color=TEXT_COLOR, fontsize=14, fontweight="bold")
ax3.tick_params(colors=TEXT_COLOR)
ax3.spines[:].set_color(GRID_COLOR)
ax3.grid(True, axis="x", color=GRID_COLOR, linewidth=0.5, alpha=0.7)
ax3.set_xlim(0, state_gpc_2028.max() * 1.18)

plt.tight_layout()
fig3.savefig("fig3_gdp_per_capita_2028.png", dpi=150,
             bbox_inches="tight", facecolor=DARK_BG)
plt.close()

# ------ Fig 4: Johor detailed (all models + their average overlaid) ← MODIFIED ------
fig4, axes4 = plt.subplots(1, 2, figsize=(16, 6))
fig4.patch.set_facecolor(DARK_BG)
fig4.suptitle("Johor — All Models + Ensemble Average (GDP & Population Forecast)",
              color=TEXT_COLOR, fontsize=14, fontweight="bold")

model_colors = ["#4FC3F7","#FF7043","#66BB6A","#FFB74D","#CE93D8"]

johor_hist = df[df["state"] == "Johor"].sort_values("year")
X_johor    = johor_hist[["year"]].values
X_fut2     = np.array(FORECAST_YEARS).reshape(-1, 1)

for ax_i, target in enumerate(["gdp_rm_million", "population"]):
    ax = axes4[ax_i]
    ax.set_facecolor(CARD_BG)
    ax.spines[:].set_color(GRID_COLOR)
    y_hist = johor_hist[target].values
    ax.plot(johor_hist["year"], y_hist, "o-", color="white",
            linewidth=3, markersize=7, zorder=5, label="Actual")

    fig4_all_preds = []   # ← collect for averaging in Fig 4
    for (mname, model), mc in zip(models.items(), model_colors):
        model.fit(X_johor, y_hist)
        preds = model.predict(X_fut2)
        fig4_all_preds.append(preds)
        ax.plot(FORECAST_YEARS, preds, "s--", color=mc, linewidth=1.5,
                markersize=5, alpha=0.6, label=mname)   # ← slightly faded

    # Plot the ensemble average line on top
    avg_line = np.mean(fig4_all_preds, axis=0)
    ax.plot(FORECAST_YEARS, avg_line, "D-", color="#FFFFFF", linewidth=2.5,
            markersize=7, zorder=6, label="Ensemble Avg")   # ← bold white line

    ax.axvline(x=2023.5, color="#555577", linewidth=1.2, linestyle=":")
    ax.set_title(target.replace("_", " ").title(), color=TEXT_COLOR, fontsize=11)
    ax.set_xlabel("Year", color=TEXT_COLOR, fontsize=9)
    ax.tick_params(colors=TEXT_COLOR)
    ax.grid(True, color=GRID_COLOR, linewidth=0.5, alpha=0.6)
    ax.legend(fontsize=7, facecolor=DARK_BG, labelcolor=TEXT_COLOR, loc="upper left")

plt.tight_layout(pad=2)
fig4.savefig("fig4_johor_models_overlay.png", dpi=150,
             bbox_inches="tight", facecolor=DARK_BG)
plt.close()

print("✅ All done!")
print(f"   Rows in output CSV  : {len(out_df)}")
print(f"   States forecast     : {len(states)}")
print(f"   Forecast years      : {FORECAST_YEARS}")
print(f"   Models evaluated    : {list(models.keys())}")
print(f"   Forecast method     : Ensemble Average (mean of all models)")

✅ All done!
   Rows in output CSV  : 224
   States forecast     : 16
   Forecast years      : [2024, 2025, 2026, 2027, 2028]
   Models evaluated    : ['Gradient Boosting', 'Random Forest', 'Extra Trees', 'Poly Ridge (deg 3)', 'ElasticNet Poly']
   Forecast method     : Ensemble Average (mean of all models)


In [35]:
# Write to sc_silver.gdp_forecast
try:
    write_table(out_df, 'sc_silver', 'gdp_forecast')
    print("Successfully written to Supabase: sc_silver.gdp_forecast")
except Exception as e:
    print(f"Failed to write to Supabase: {e}")

Table sc_silver.gdp_forecast written successfully.
Successfully written to Supabase: sc_silver.gdp_forecast
